In [1]:
import pandas as pd
import re
import os
from fuzzywuzzy import fuzz
from openpyxl import load_workbook
from openpyxl.styles import PatternFill
from tqdm import tqdm
from datetime import datetime

In [2]:
# === CONFIG ===

# Input study file + sheet
STUDY_FILE = './out/HDP01233_combined_DD_2026-04-16_matches confirmed.xlsx'
STUDY_SHEET = 'EnhancedDD'

# Study variable columns
ENCODING_COLUMN = 'Choices, Calculations, OR Slider Labels'
FIELD_LABEL_COLUMN = 'Field Label'
VARIABLE_NAME_COLUMN = 'Variable / Field Name'

# HEAL CDE knowledge base
CDE_FILE = './KnowledgeBase/Compiled_CORE_CDEs list_English_one sheet_as of 2025-01-28.xlsx'

# --- Deterministic output directory ---
# Works in BOTH notebooks and .py scripts
try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    # __file__ is not defined in notebooks / interactive sessions
    BASE_DIR = os.getcwd()

OUTPUT_DIR = os.path.join(BASE_DIR, "out")
print("📁 Output directory (absolute):", os.path.abspath(OUTPUT_DIR))

# --- CRF-aware matching settings ---
EXPECTED_CRF_COL = "HEAL Core CRF Match"
RESTRICT_TO_EXPECTED_CRF_FIRST = True
FALLBACK_TO_ALL_IF_BELOW = 70
CRF_MAP_THRESHOLD = 85

# --- Configurable confidence thresholds ---
CONFIDENCE_THRESHOLDS = {
    'high': 80,
    'medium': 51,
    'minimum_score': 30  # Don't show matches below this threshold
}


📁 Output directory (absolute): c:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out


In [3]:
# Weighting for different text components
TEXT_WEIGHT = 0.4
ENCODING_WEIGHT = 0.6

def normalize_string(s):
    """Normalize string by converting to lowercase, handling NaN or float, and keeping spaces."""
    if isinstance(s, str):
        s = re.sub(r'[^a-zA-Z0-9\s=]', '', s.lower())  # Keep letters, numbers, spaces, and equal signs
        s = re.sub(r'\s+', ' ', s)  # Collapse multiple spaces into one
        return s.strip()
    else:
        return ''

# --- NEW: CRF label normalization + KB mapping helpers ---
def normalize_crf_label(s: str) -> str:
    """Normalize CRF labels so LLM/study variants map cleanly to KB CRF names."""
    if not isinstance(s, str):
        return ""
    s = s.strip().lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()

    # remove generic words LLMs often add
    junk = {"questionnaire", "scale", "inventory", "form", "survey", "assessment"}
    tokens = [t for t in s.split() if t not in junk]
    return " ".join(tokens)

def build_crf_mapper(kb_crf_names):
    """
    Returns a function that maps arbitrary CRF labels to the closest KB CRF Name.
    Uses normalization + fuzzy matching fallback.
    """
    kb = [c for c in kb_crf_names if isinstance(c, str) and c.strip()]
    kb_norm = {c: normalize_crf_label(c) for c in kb}

    def map_to_kb(label: str, threshold: int = CRF_MAP_THRESHOLD):
        raw = label if isinstance(label, str) else ""
        n = normalize_crf_label(raw)
        if not n:
            return None

        # exact match on normalized form
        for kb_label, kb_n in kb_norm.items():
            if n == kb_n:
                return kb_label

        # fuzzy fallback
        best_label, best_score = None, -1
        for kb_label, kb_n in kb_norm.items():
            score = fuzz.token_set_ratio(n, kb_n)
            if score > best_score:
                best_score = score
                best_label = kb_label

        return best_label if best_score >= threshold else None

    return map_to_kb


def similarity_score(str1, str2):
    """
    Calculate token-based fuzzy similarity between two strings.
    100% means an exact match, 0% means completely different.
    """
    return fuzz.token_set_ratio(str1, str2)

def enhanced_similarity_score(study_text, study_encoding, cde_text, cde_encoding):
    """
    Enhanced similarity scoring that weights encodings and text differently.
    Encodings are often more standardized, so they get higher weight.
    """
    # Handle cases where components might be empty
    if not study_encoding and not cde_encoding:
        return similarity_score(study_text, cde_text)
    if not study_text and not cde_text:
        return similarity_score(study_encoding, cde_encoding)

    text_score = similarity_score(study_text, cde_text) if study_text and cde_text else 0
    encoding_score = similarity_score(study_encoding, cde_encoding) if study_encoding and cde_encoding else 0

    # Weighted average
    return (text_score * TEXT_WEIGHT) + (encoding_score * ENCODING_WEIGHT)

def print_matching_summary(final_df):
    """Print summary statistics of the matching process."""
    processed_df = final_df[final_df['Best Match CDE Name'].notna()]
    total_processed = len(processed_df)

    if total_processed == 0:
        print("📊 No matches found.")
        return

    high_conf = len(processed_df[processed_df['Best Match Score'] >= CONFIDENCE_THRESHOLDS['high']])
    medium_conf = len(processed_df[
        (processed_df['Best Match Score'] >= CONFIDENCE_THRESHOLDS['medium']) &
        (processed_df['Best Match Score'] < CONFIDENCE_THRESHOLDS['high'])
    ])
    low_conf = len(processed_df[processed_df['Best Match Score'] < CONFIDENCE_THRESHOLDS['medium']])

    print(f"\n📊 Matching Summary:")
    print(f"   Total variables processed: {total_processed}")
    print(f"   🟢 High confidence matches (≥{CONFIDENCE_THRESHOLDS['high']}%): {high_conf} ({high_conf/total_processed*100:.1f}%)")
    print(f"   🟧 Medium confidence matches ({CONFIDENCE_THRESHOLDS['medium']}-{CONFIDENCE_THRESHOLDS['high']-1}%): {medium_conf} ({medium_conf/total_processed*100:.1f}%)")
    print(f"   🟥 Low confidence matches (<{CONFIDENCE_THRESHOLDS['medium']}%): {low_conf} ({low_conf/total_processed*100:.1f}%)")


def compare_encodings(
    study_file,
    encoding_column='encodings',
    field_label_column='field_label',
    cde_file='./KnowledgeBase/Compiled_CORE_CDEs list_English_one sheet_as of 2025-01-28.xlsx',
    study_sheet='Sheet1',
    variable_name_column='name',   # 👈 new
    dry_run=False
):
    """
    Compare study data dictionary encodings and field labels with HEAL CDE encodings using fuzzy token-based similarity.
    Returns path to the output file.

    NEW BEHAVIOR:
    - Best Match columns are CRF-FIRST ONLY (restricted to Expected CRF)
    - Potential Match 2/3 are ALWAYS global fallback suggestions (all CDEs)
    """
    # Ensure output directory exists
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # ✅ Step 2A: show where OUTPUT_DIR resolves (absolute)
    print("📁 Output directory (absolute):", os.path.abspath(OUTPUT_DIR))

    # Load study data
    print("📂 Loading study data...")
    if study_file.endswith('.xlsx'):
        full_study_df = pd.read_excel(study_file, sheet_name=study_sheet)
    else:
        full_study_df = pd.read_csv(study_file)

    # Filter out 'No CRF match' rows
    if EXPECTED_CRF_COL in full_study_df.columns:
        skipped_df = full_study_df[full_study_df[EXPECTED_CRF_COL] == 'No CRF match'].copy()
        study_df   = full_study_df[full_study_df[EXPECTED_CRF_COL] != 'No CRF match'].copy()
        print(f"✅ Processing {len(study_df)} rows; skipping {len(skipped_df)} rows (No CRF match).")
    else:
        skipped_df = pd.DataFrame()
        study_df   = full_study_df.copy()
        print(f"✅ Processing all {len(study_df)} rows (no '{EXPECTED_CRF_COL}' column found).")

    if dry_run:
        print(f"🔍 DRY RUN: Would process {len(study_df)} variables against HEAL CDEs")
        return None

    # --- Detect the variable-name column robustly ---
    candidate_var_cols = [
        variable_name_column, 'Variable / Field Name', 'Variable/Field Name',
        'field_name', 'Field Name', 'variable_name', 'name', 'variable'
    ]
    var_col = next((c for c in candidate_var_cols if c in study_df.columns), None)
    if var_col is None:
        print("⚠️ Could not find a variable-name column. Low_Confidence_Analysis will show 'Unknown'.")
    else:
        print(f"🔎 Using '{var_col}' as the study variable-name column.")

    # Initialize new columns for CDE matches
    new_cols = [
        'Best Match CDE Name', 'Best Match Score', 'Best Match CRF Name',
        'Potential Match 2 - CDE Name', 'Potential Match 2 - Score', 'Potential Match 2 - CRF Name',
        'Potential Match 3 - CDE Name', 'Potential Match 3 - Score', 'Potential Match 3 - CRF Name'
    ]
    for col in new_cols:
        if col not in study_df.columns:
            study_df[col] = None
        else:
            study_df[col] = None  # reset

    # Normalize study encodings and field labels
    print("🔧 Normalizing study variables...")
    study_df['Normalized Text'] = study_df[field_label_column].apply(normalize_string)
    study_df['Normalized Encoding'] = study_df[encoding_column].apply(normalize_string)
    study_df['Normalized Combined'] = study_df.apply(
        lambda row: normalize_string(f"{row.get(encoding_column, '')} | {row.get(field_label_column, '')}")
        if pd.notna(row.get(encoding_column)) or pd.notna(row.get(field_label_column)) else '',
        axis=1
    )

    # Load and normalize HEAL CDEs
    print("📚 Loading HEAL CDE database...")
    cde_df = pd.read_excel(cde_file, sheet_name='ALL')
    cde_df = cde_df.dropna(subset=['PV Description', 'Additional Notes (Question Text)'])

    # --- sanitize KB CRF Name + build CRF mapper ---
    cde_df["CRF Name"] = cde_df["CRF Name"].astype(str).str.strip()
    kb_crf_names = sorted(cde_df["CRF Name"].dropna().unique())
    map_crf = build_crf_mapper(kb_crf_names)

    cde_df['Normalized Text'] = cde_df['Additional Notes (Question Text)'].apply(normalize_string)
    cde_df['Normalized Encoding'] = cde_df['PV Description'].apply(normalize_string)

    print(f"✅ Loaded {len(cde_df)} HEAL CDEs for comparison.")

    # Track low-confidence matches (for primary/best match only, as before)
    low_confidence_matches = []

    # Matching
    print("🔍 Running CRF-first best match + global fallback suggestions...")
    for idx, row in tqdm(study_df.iterrows(), total=len(study_df), desc="Matching variables", unit="vars"):

        if not row.get('Normalized Combined', ''):
            continue

        # --------------- NEW: determine expected KB CRF ---------------
        expected_raw = row.get(EXPECTED_CRF_COL, None)
        expected_kb = map_crf(expected_raw) if expected_raw else None

        # Primary (CRF-first only)
        best_match = None
        best_score = -1
        best_crf_name = None

        # Fallback pool (global)
        global_matches = []  # (cde_var, score, crf_name)

        # Single-pass scoring over KB
        for _, cde_row in cde_df.iterrows():

            score_percentage = enhanced_similarity_score(
                row['Normalized Text'],
                row['Normalized Encoding'],
                cde_row['Normalized Text'],
                cde_row['Normalized Encoding']
            )

            if score_percentage < CONFIDENCE_THRESHOLDS['minimum_score']:
                continue

            cde_var = cde_row['Variable Name']
            cde_crf = cde_row['CRF Name']

            # Always collect global candidates for fallback columns
            global_matches.append((cde_var, score_percentage, cde_crf))

            # CRF-first best match: only consider candidates inside expected KB CRF
            if RESTRICT_TO_EXPECTED_CRF_FIRST and expected_kb:
                if cde_crf == expected_kb and score_percentage > best_score:
                    best_match = cde_var
                    best_score = score_percentage
                    best_crf_name = cde_crf
            else:
                # If CRF restriction is OFF or we couldn't map expected CRF,
                # we do NOT promote global matches into Best Match (by your design).
                pass

        # --------------- write primary/best match (CRF-first only) ---------------
        if best_match is not None:
            study_df.at[idx, 'Best Match CDE Name'] = best_match
            study_df.at[idx, 'Best Match Score'] = round(best_score, 1)
            study_df.at[idx, 'Best Match CRF Name'] = best_crf_name

            # low-confidence tracker (still tied to primary match)
            if best_score < CONFIDENCE_THRESHOLDS['high']:
                study_text_raw = f"{row.get(field_label_column, '')} | {row.get(encoding_column, '')}".strip()
                if len(study_text_raw) > 120:
                    study_text_raw = study_text_raw[:117] + "..."
                low_confidence_matches.append({
                    'Row': (idx + 2),
                    'Study_Variable': (row.get(var_col) if var_col else 'Unknown'),
                    'Study_Text': study_text_raw,
                    'Best_CDE': best_match,
                    'Score': round(best_score, 1),
                    'CRF_Name': best_crf_name,
                    'Expected_CRF': expected_raw,
                    'Expected_CRF_Mapped_To_KB': expected_kb
                })

        # --------------- write fallback suggestions into Potential 2/3 ---------------
        # sort global by score desc, then de-dupe by variable name
        global_matches.sort(key=lambda x: x[1], reverse=True)

        seen = set()
        unique_global = []
        for name, score, crf in global_matches:
            if name in seen:
                continue
            # Optional: don't repeat the primary best match in the fallback slots
            if best_match is not None and name == best_match:
                continue
            unique_global.append((name, score, crf))
            seen.add(name)

        # Fill Potential Match 2 & 3 from top global suggestions
        for slot_i, (match_name, match_score, crf_name) in enumerate(unique_global[:2], start=2):
            study_df.at[idx, f'Potential Match {slot_i} - CDE Name'] = match_name
            study_df.at[idx, f'Potential Match {slot_i} - Score'] = round(match_score, 1)
            study_df.at[idx, f'Potential Match {slot_i} - CRF Name'] = crf_name

    # ----------------------------
    # 1) Low-confidence sheet DF
    # ----------------------------
    low_conf_df = pd.DataFrame(low_confidence_matches) if low_confidence_matches else pd.DataFrame()

    # ----------------------------
    # 2) Merge skipped rows back in (so final output includes all original rows)
    # ----------------------------
    for col in new_cols:
        if col not in skipped_df.columns:
            skipped_df[col] = None

    for col in ["Normalized Text", "Normalized Encoding", "Normalized Combined"]:
        if col not in skipped_df.columns:
            skipped_df[col] = None

    final_df = pd.concat([study_df, skipped_df], ignore_index=True)

    # ----------------------------
    # 3) Build output filename + print exact path (Step 2B)
    # ----------------------------
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_base = os.path.basename(study_file).rsplit(".", 1)[0]
    output_file = os.path.join(OUTPUT_DIR, f"{output_base}_vlmd_cdesearch_{timestamp}.xlsx")

    print("📄 Writing output file to (absolute path):", os.path.abspath(output_file))

    # ----------------------------
    # 4) Write Excel outputs
    # ----------------------------
    with pd.ExcelWriter(output_file, engine="xlsxwriter") as writer:
        final_df.to_excel(writer, sheet_name="VLMD_Results", index=False)

        if not low_conf_df.empty:
            low_conf_df.to_excel(writer, sheet_name="Low_Confidence_Analysis", index=False)
            print(f"📝 {len(low_conf_df)} low-confidence matches saved for analysis")
        else:
            print("📝 No Low_Confidence_Analysis sheet (all matches high-confidence or none found).")

    print(f"💾 CDE matching complete. Results saved to {output_file}")
    print_matching_summary(final_df)

    return output_file


def apply_color_coding(output_file):
    """
    Apply color coding to the Excel file based on confidence levels:
    - Green (≥80): High confidence
    - Orange (51–79): Medium confidence
    - Red (<51): Low confidence
    """
    print("\n🎨 Applying color coding and confidence levels...")

    wb = load_workbook(output_file)
    ws = wb['VLMD_Results'] if 'VLMD_Results' in wb.sheetnames else wb.active

    # ---- Find header columns ----
    headers = [cell.value for cell in ws[1]]

    def find_col(header_name):
        for i, v in enumerate(headers, start=1):
            if isinstance(v, str) and v.strip() == header_name:
                return i
        return None

    best_match_score_col = find_col("Best Match Score")
    if not best_match_score_col:
        print("⚠️ Could not find 'Best Match Score' column. Skipping color coding.")
        wb.save(output_file)
        return

    # If "Confidence Level" already exists, reuse it; otherwise insert next to score
    confidence_col_idx = find_col("Confidence Level")
    if confidence_col_idx is None:
        confidence_col_idx = best_match_score_col + 1
        ws.insert_cols(confidence_col_idx)
        ws.cell(row=1, column=confidence_col_idx).value = "Confidence Level"
        # refresh headers after insertion
        headers = [cell.value for cell in ws[1]]

    # ---- Use ARGB colors (alpha + rgb) ----
    FILL_GREEN  = PatternFill(fill_type="solid", fgColor="FFC6EFCE")
    FILL_ORANGE = PatternFill(fill_type="solid", fgColor="FFFFEB9C")
    FILL_RED    = PatternFill(fill_type="solid", fgColor="FFFFC7CE")

    # ---- Apply per-row ----
    for row_idx in range(2, ws.max_row + 1):
        score = ws.cell(row=row_idx, column=best_match_score_col).value
        confidence_cell = ws.cell(row=row_idx, column=confidence_col_idx)

        if score is None or score == "":
            continue

        # Make sure score is numeric (handles cases where Excel stores it as text)
        try:
            score_num = float(score)
        except Exception:
            # If it can't be parsed, skip coloring but leave a breadcrumb
            confidence_cell.value = "Unparseable score"
            continue

        score_cell = ws.cell(row=row_idx, column=best_match_score_col)

        if score_num >= CONFIDENCE_THRESHOLDS['high']:
            confidence_cell.value = "High confidence"
            score_cell.fill = FILL_GREEN
        elif score_num >= CONFIDENCE_THRESHOLDS['medium']:
            confidence_cell.value = "Medium confidence"
            score_cell.fill = FILL_ORANGE
        else:
            confidence_cell.value = "Low confidence"
            score_cell.fill = FILL_RED

    wb.save(output_file)
    print("✨ Color coding applied successfully!")


In [4]:
def main(dry_run=False):
    """Main function to run the VLMD CDE matching process."""

    print("🚀 Starting HEAL CDE Variable Level Metadata Matching")
    print(
        f"📋 Configuration: High confidence ≥{CONFIDENCE_THRESHOLDS['high']}%, "
        f"Medium ≥{CONFIDENCE_THRESHOLDS['medium']}%, "
        f"Minimum threshold ≥{CONFIDENCE_THRESHOLDS['minimum_score']}%"
    )

    # --- Echo CRF-aware matching settings ---
    print("\n🧭 CRF-aware matching settings:")
    print(f"   Expected CRF column           : {EXPECTED_CRF_COL}")
    print(f"   Restrict to expected CRF first: {RESTRICT_TO_EXPECTED_CRF_FIRST}")
    print(f"   CRF mapping threshold          : {CRF_MAP_THRESHOLD}")
    print("   (Fallback suggestions will still populate Potential Match 2/3.)")

    # Optional: quick preflight column check (non-fatal)
    try:
        preview_df = pd.read_excel(STUDY_FILE, sheet_name=STUDY_SHEET, nrows=5)
        if EXPECTED_CRF_COL not in preview_df.columns:
            print(
                f"\n⚠️  Warning: Expected CRF column '{EXPECTED_CRF_COL}' not found in '{STUDY_SHEET}'.\n"
                "    Best Match will not be CRF-restricted; fallback suggestions may still appear.\n"
            )
    except Exception as e:
        print(f"\n⚠️  Preflight check skipped (could not read file): {e}\n")

    # Run CDE matching (writes file + returns path)
    output_file = compare_encodings(
        STUDY_FILE,
        encoding_column=ENCODING_COLUMN,
        field_label_column=FIELD_LABEL_COLUMN,
        cde_file=CDE_FILE,
        study_sheet=STUDY_SHEET,
        variable_name_column=VARIABLE_NAME_COLUMN,
        dry_run=dry_run
    )

    if dry_run:
        print("\n🧪 Dry run complete (no output file written).")
        return

    if not output_file:
        print("\n❌ No output file was returned. Check logs above for any errors.")
        return

    print("\n✅ Output file created:")
    print("   📄", os.path.abspath(output_file))

    # Apply color coding (fail-safe)
    try:
        apply_color_coding(output_file)
    except Exception as e:
        print(f"\n⚠️  Color coding failed (file is still valid): {e}")

    print(f"\n🎉 VLMD CDE matching complete!")
    print(f"📊 Results saved to: {output_file}")
    print(f"📋 Review the 'Low_Confidence_Analysis' sheet to improve future matching")


In [5]:
main(dry_run=False)

🚀 Starting HEAL CDE Variable Level Metadata Matching
📋 Configuration: High confidence ≥80%, Medium ≥51%, Minimum threshold ≥30%

🧭 CRF-aware matching settings:
   Expected CRF column           : HEAL Core CRF Match
   Restrict to expected CRF first: True
   CRF mapping threshold          : 85
   (Fallback suggestions will still populate Potential Match 2/3.)
📁 Output directory (absolute): c:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out
📂 Loading study data...
✅ Processing 85 rows; skipping 97 rows (No CRF match).
🔎 Using 'Variable / Field Name' as the study variable-name column.
🔧 Normalizing study variables...
📚 Loading HEAL CDE database...
✅ Loaded 346 HEAL CDEs for comparison.
🔍 Running CRF-first best match + global fallback suggestions...


Matching variables: 100%|██████████| 85/85 [00:03<00:00, 27.33vars/s]


📄 Writing output file to (absolute path): c:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_cdesearch_20260416_144204.xlsx
📝 25 low-confidence matches saved for analysis
💾 CDE matching complete. Results saved to c:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_cdesearch_20260416_144204.xlsx

📊 Matching Summary:
   Total variables processed: 34
   🟢 High confidence matches (≥80%): 9 (26.5%)
   🟧 Medium confidence matches (51-79%): 19 (55.9%)
   🟥 Low confidence matches (<51%): 6 (17.6%)

✅ Output file created:
   📄 c:\Users\lmaefos\Code Stuffs\CDE_detective\CDE_ID_detective_revamp\out\HDP01233_combined_DD_2026-04-16_matches confirmed_vlmd_cdesearch_20260416_144204.xlsx

🎨 Applying color coding and confidence levels...
✨ Color coding applied successfully!

🎉 VLMD CDE matching complete!
📊 Results saved to: c:\Users\lmaefos\Code Stuffs\CDE

# HEAL CDE Fuzzy Matching Script

This script automates the matching of study data dictionaries to the HEAL Core Common Data Elements (CDEs), using smart fuzzy text comparison and a color-coded Excel output for easier review.

It is designed to help identify the **Best Match** and **Top 2 Potential Matches** for each study variable — even if the wording or encoding order is slightly different from the CDE standard.

---

## ✨ Key Features
- **Fuzzy Matching**: Uses token-based similarity (`Token Set Ratio`) to handle small typos and different word orders.
- **Normalized Comparisons**: Cleans and standardizes text for reliable matching.
- **Separate Output Folder**: All results are saved neatly into an `/out/` subfolder.
- **Color Coded Scores**:  
  - 🟩 **Green** for matches ≥ 80%  
  - 🟧 **Orange** for matches 51–79%  
  - 🟥 **Red** for matches ≤ 50%
- **No Duplicate Matches**: Ensures Best Match and Potential Matches are truly different.

---

## 🚀 How It Works

1. **Input**:
   - A study data dictionary (Excel `.xlsx` or CSV `.csv` file).
   - The master HEAL CDEs file (Excel file).

2. **Process**:
   - Normalize (clean) text by lowercasing, removing special characters, and preserving logical structures like equal signs.
   - Compare the study's "Encoding + Field Label" to the CDE's "PV Description + Question Text".
   - Select the best match and top two alternatives based on fuzzy matching scores.
   - Color-code the best match scores for easy visualization.

3. **Output**:
   - A new Excel file saved into `/out/`, with best matches listed and scores color-coded.

---

## 🛠️ Requirements

Install the following Python packages:

```bash
pip install pandas openpyxl fuzzywuzzy
```

---

## 📂 Folder Structure

```
/in/         # Input study files
/out/        # Output matched files (automatically created if not present)
/KnowledgeBase/ # Contains the HEAL Core CDE file
script.py    # Your main script
```

---

## 📋 Usage Example

```bash
python script.py
```

The output file will appear in the `/out/` folder and will be named something like:

```
SAMPLE_sprint_2020-12-16_vlmd_cdesearch.xlsx
```

---

## 🧠 Notes

- The script **requires** the correct columns to be named in the study file (e.g., `Choices, Calculations, OR Slider Labels` and `Field Label`).
- Only the **Best Match Score** column is color coded for quick review.
- Ensure your HEAL CDE master file contains the necessary columns: `PV Description` and `Additional Notes (Question Text)`.


# Notes for improvement
- copyrighted CDEs don't populate a 'Best Match CDE Name' because the knowledge base is empty for fields in the permissible values for copyrighted CDEs. 
- top 3 matches populate, sometimes there's no "Best Match", but there is a value for "Potential Matches" (the best match is missing)
- CRF level matches appear to work well
- VLMD matches do not appear to work well
- What if i add an additional assistant that will review each row, read through the following:
    - original variable name, original form name, original description, canonical CRF name rationale, HEAL Core CRF Match rationale, and identify the best Form name and CDE match 
